# Graph Diffusion Training (Kaggle)
文本条件离散扩散图结构生成

In [ ]:
import os
REPO_DIR = '/kaggle/working/PlanDiffusion_wzm'
if not os.path.exists(REPO_DIR):
    os.system(f'git clone -b wzm-shenzhou https://github.com/WeeZHnMin/PlanDiffusion_wzm.git {REPO_DIR}')
else:
    os.system(f'git -C {REPO_DIR} pull origin wzm-shenzhou')
os.chdir(REPO_DIR)
print('repo ready:', REPO_DIR)

In [ ]:
import subprocess
subprocess.run(['pip', 'install', '-q', 'tokenizers'], check=True)
print('依赖安装完成')

## 配置

In [ ]:
DATA_PATH  = '/kaggle/input/datasets/weizhiming/graph-diffusion/graph_dataset.npz'
VOCAB_PATH = '/kaggle/working/PlanDiffusion_wzm/node_diffusion/unified_vocab/vocab_config.json'
SAVE_DIR   = '/kaggle/working/checkpoints/graph_diffusion'
RESUME     = ''   # 续训时填写路径，例如 '/kaggle/input/.../best.pt'

BATCH_SIZE   = 256
TOTAL_STEPS  = 500_000
LR           = 1e-4
WEIGHT_DECAY = 1e-4
LOG_EVERY    = 200
SAVE_EVERY   = 10_000
TIMESTEPS    = 500

N_LAYERS = 6
DX       = 256
DE       = 64
DY       = 256
N_HEAD   = 8
DROPOUT  = 0.1

## 导入 & 环境

In [ ]:
import sys, gc, json, time, warnings
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW

sys.path.insert(0, REPO_DIR)
from graph_diffusion.dataset   import make_loader
from graph_diffusion.model     import GraphTransformer
from graph_diffusion.diffusion import GaussianNoiseSchedule, DiscreteUniformTransition, apply_noise

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.backends.cudnn.benchmark = True
print(f'device: {device}  |  GPU 数量: {torch.cuda.device_count()}')

## 损失函数

In [ ]:
def compute_loss_and_acc(pred_X, pred_E, true_X, true_E, node_mask):
    B, N, Kx = pred_X.shape
    Ke = pred_E.shape[-1]
    x_mask    = node_mask.float()
    e_mask    = (node_mask.unsqueeze(2) * node_mask.unsqueeze(1)).float()
    triu      = torch.triu(torch.ones(N, N, device=pred_E.device, dtype=torch.bool), diagonal=1)
    triu_mask = e_mask * triu.unsqueeze(0)

    true_X_idx = true_X.argmax(-1)
    true_E_idx = true_E.argmax(-1)

    loss_x = F.cross_entropy(
        pred_X.reshape(B * N, Kx), true_X_idx.reshape(B * N), reduction='none'
    ).reshape(B, N)
    loss_x = (loss_x * x_mask).sum() / (x_mask.sum() + 1e-8)

    loss_e = F.cross_entropy(
        pred_E.reshape(B * N * N, Ke), true_E_idx.reshape(B * N * N), reduction='none'
    ).reshape(B, N, N)
    loss_e = (loss_e * triu_mask).sum() / (triu_mask.sum() + 1e-8)

    with torch.no_grad():
        acc_x    = ((pred_X.argmax(-1) == true_X_idx).float() * x_mask).sum() / (x_mask.sum() + 1e-8)
        actual   = (true_E_idx == 1).float() * triu_mask
        correct  = ((pred_E.argmax(-1) == 1) & (true_E_idx == 1)).float() * triu_mask
        recall_e = correct.sum() / (actual.sum() + 1e-8)

    return loss_x, loss_e, acc_x.item(), recall_e.item()

## 数据 / 模型 / 优化器

In [ ]:
Path(SAVE_DIR).mkdir(parents=True, exist_ok=True)

vocab_cfg = json.loads(open(VOCAB_PATH, encoding='utf-8').read())
bpe_vocab = vocab_cfg['bpe_vocab_size']

loader = make_loader(DATA_PATH, BATCH_SIZE, shuffle=True, num_workers=4)
print(f'数据集: {len(loader.dataset)} 条  batch={BATCH_SIZE}  steps/epoch={len(loader.dataset)//BATCH_SIZE}')

model = GraphTransformer(
    x_classes=32, e_classes=2, bpe_vocab_size=bpe_vocab, text_embed_dim=128,
    n_layers=N_LAYERS, dx=DX, de=DE, dy=DY, n_head=N_HEAD, dropout=DROPOUT,
).to(device)

n_gpus = torch.cuda.device_count()
if n_gpus > 1:
    model = nn.DataParallel(model)
    print(f'DataParallel: {n_gpus} 张 GPU')
print(f'参数量: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M')

schedule   = GaussianNoiseSchedule(T=TIMESTEPS)
transition = DiscreteUniformTransition(x_classes=32, e_classes=2)

opt    = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY, betas=(0.9, 0.95))
scaler = torch.amp.GradScaler('cuda')
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=TOTAL_STEPS, eta_min=LR * 0.1)

## 续训（可选）

In [ ]:
start_step = 0
best_loss  = float('inf')

if RESUME and Path(RESUME).exists():
    ckpt = torch.load(RESUME, map_location=device)
    sd   = {k.replace('module.', '').replace('_orig_mod.', ''): v for k, v in ckpt['model'].items()}
    raw  = model.module if hasattr(model, 'module') else model
    raw.load_state_dict(sd, strict=True)
    opt.load_state_dict(ckpt['opt'])
    scaler.load_state_dict(ckpt['scaler'])
    if 'scheduler' in ckpt:
        scheduler.load_state_dict(ckpt['scheduler'])
    start_step = ckpt['step'] + 1
    best_loss  = ckpt.get('best_loss', float('inf'))
    print(f'resumed from step {start_step} / {TOTAL_STEPS}')
else:
    print('从零开始训练')

## 训练

In [ ]:
log_mode = 'a' if RESUME and Path(RESUME).exists() else 'w'
log_file = open(Path(SAVE_DIR) / 'train_log.jsonl', log_mode, encoding='utf-8', buffering=1)

def infinite(loader):
    while True:
        yield from loader

data_iter     = infinite(loader)
running_loss  = running_loss_x = running_loss_e = 0.0
running_acc_x = running_acc_e  = 0.0
save_win_loss = save_win_steps = 0
t0 = time.perf_counter()

model.train()
for step in range(start_step, TOTAL_STEPS):
    X, E, node_mask, ptokens, plens = next(data_iter)
    X         = X.to(device)
    E         = E.to(device)
    node_mask = node_mask.to(device)
    ptokens   = ptokens.to(device)
    plens     = plens.to(device) if isinstance(plens, torch.Tensor) else torch.tensor(plens).to(device)

    B       = X.shape[0]
    t_int   = torch.randint(1, TIMESTEPS + 1, (B,), device=device)
    t_float = t_int.float() / TIMESTEPS
    Xt, Et, _ = apply_noise(X, E, node_mask, t_int - 1, schedule, transition, device)

    opt.zero_grad()
    with torch.autocast(device_type='cuda', dtype=torch.float16):
        pred_X, pred_E = model(Xt, Et, node_mask, ptokens, plens, t_float)
        loss_x, loss_e, acc_x, recall_e = compute_loss_and_acc(pred_X, pred_E, X, E, node_mask)
        loss = loss_x + 5.0 * loss_e

    scaler.scale(loss).backward()
    scaler.unscale_(opt)
    nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    scaler.step(opt)
    scaler.update()
    scheduler.step()

    lv             = loss.item()
    running_loss   += lv
    running_loss_x += loss_x.item()
    running_loss_e += loss_e.item()
    running_acc_x  += acc_x
    running_acc_e  += recall_e
    save_win_loss  += lv
    save_win_steps += 1

    if step % 100 == 0:
        torch.cuda.empty_cache()
        gc.collect()

    if step % LOG_EVERY == 0 and step > 0:
        n   = LOG_EVERY
        avg = {
            'loss':     running_loss   / n,
            'loss_x':   running_loss_x / n,
            'loss_e':   running_loss_e / n,
            'acc_x':    running_acc_x  / n,
            'recall_e': running_acc_e  / n,
        }
        running_loss = running_loss_x = running_loss_e = running_acc_x = running_acc_e = 0.0
        elapsed = time.perf_counter() - t0
        t0 = time.perf_counter()
        lr_now = scheduler.get_last_lr()[0]
        print(f'step {step:7d} | loss {avg["loss"]:.4f} '
              f'| loss_x {avg["loss_x"]:.4f} acc_x {avg["acc_x"]:.3f} '
              f'| loss_e {avg["loss_e"]:.4f} recall_e {avg["recall_e"]:.3f} '
              f'| lr {lr_now:.2e} | {elapsed:.1f}s')
        log_file.write(json.dumps({
            'step': step, **{k: round(v, 4) for k, v in avg.items()},
            'lr': round(lr_now, 8), 'elapsed': round(elapsed, 1),
        }, ensure_ascii=False) + '\n')

    if step % SAVE_EVERY == 0 and step > 0:
        win_avg        = save_win_loss / max(save_win_steps, 1)
        save_win_loss  = save_win_steps = 0
        if win_avg < best_loss:
            best_loss = win_avg
            raw = model.module if hasattr(model, 'module') else model
            torch.save({
                'model':     raw.state_dict(),
                'opt':       opt.state_dict(),
                'scaler':    scaler.state_dict(),
                'scheduler': scheduler.state_dict(),
                'step':      step,
                'best_loss': best_loss,
            }, Path(SAVE_DIR) / 'best.pt')
            print(f'  best saved → step={step} loss={best_loss:.4f}')

log_file.close()
print('训练完成')